## Eftychia Kazasi - 2/9/2026

*For the report I tried to use the weighted versions of the functions*

### Step 1: Load the dataset

In [ ]:
import pandas as pd
import numpy as np

air_pollution = pd.read_csv("/.../pollution_cleaneddata.csv") # adjust the path to your file

MORT_y = air_pollution['MORT'].values # print as array/vector
POOR_x = air_pollution['POOR'].values

### Step 2: Find the k nearest points for each value $x_0$ of POOR_x

In [13]:
def find_k_nearest_points(x, x0, k):

    distances = np.abs(x - x0)
    idx = np.argsort(distances)[:k] # sort the distances in ascending order and keep the indices of the k nearest points
    
    return idx, distances[idx] # return the indices of the k nearest points and their corresponding distances

In case of two or more distances being exactly equal, the numpy function *argsort* keeps the points that appear earlier in the x array. 

### Step 3: Assign a weight to each of the *k* nearest points based on their distance from $x_0$

For this step, we have to calculate the weights that are assigned to the *k* nearest points selected in step 2, based on how close each point is to $x_0$. The closest the point, the higher the weight should be for this point. After searching online, one method used frequently for calculating these weights is the **gaussian kernel function**, that follows this formula:

$weights = \exp\left(-\frac{(x^{(i)} - x)^2}{2h^2}\right) = \exp\left(-\frac{d^2}{2h^2}\right)$, with **d** being the distance of the neighbor $x_i$ and $x_0$ and **h** being a bandwidth parameter that controls how quickly the weight decreases with distance.

In [14]:
def assign_weights(distances):

    h = distances.max()
    weights = np.exp(-(distances ** 2) / (2 * (h ** 2)))
    
    return weights

### Step 4: Fit a weighted simple OLS regression by minimizing the weighted sum of squared errors $Q(\beta_0,\beta_1) = \sum_{i=1}^{k} w_i (y_i - \beta_0 - \beta_1 x_i)^2$.

By setting the partial derivatives of *Q* with respect to $β_0$ and $β_1$ to zero, we derive the following formulas for $\bar{x}$ and $\bar{y}$ and $β_0$ and $β_1$. The equations for $β_1$ and $β_0$ are presented in the lecture in the Ordinary Least Squares (OLS) regression section without weights. For the report I adapted the equations to include the weights.

From the notes, we know $\beta_0 = \bar{y_w} - \beta_1 \bar{x_w}$ and after doing the math of solving the partial derivative with respect to $β_0$ we get that $\beta_0 = \frac{\sum_i w_i y_i}{\sum_i w_i} - \beta_1 \frac{\sum_i w_i x_i}{\sum_i w_i}$. 

So, $\bar{y}_w = \frac{\sum_i w_i y_i}{\sum_i w_i}$ and $\bar{x}_w = \frac{\sum_i w_i x_i}{\sum_i w_i}$

From solving the other parial derivative with respect to $β_1$ we get the function for $\beta_1 = \frac{\sum_i w_i (x_i - \bar{x}_w)(y_i - \bar{y}_w)}{\sum_i w_i (x_i - \bar{x}_w)^2}$


*In order to find out how we came up with the functions of $β_0$ and $β_1$ I searched online the math behind and then transfered the functions into code*. ​ 




In [15]:
def weighted_OLS(x_i, y_i, w):

    x_bar_w = np.sum(w * x_i) / np.sum(w)
    y_bar_w = np.sum(w * y_i) / np.sum(w)

    Sxx_w = np.sum(w * (x_i - x_bar_w) ** 2)
    Sxy_w = np.sum(w * (x_i - x_bar_w) * (y_i - y_bar_w))

    b1 = Sxy_w / Sxx_w
    b0 = y_bar_w - b1 * x_bar_w
    
    return b0, b1

### Step 5: Calculate the variance of the errors $σ^2$, which is the variance of the residuals

The formula for $σ^2$ is given by the following function $\frac{\sum_i w_i (y_i - \beta_0 - \beta_1 x_i)^2}{n - 2}$ based on the lecture notes. As n, which is the sample size, we are going to use the *k* and we use "-2" because 2 degrees of freedom are used to estimate $β_0$ and $β_1$.

In [16]:
def error(x_i, y_i, w, b0, b1, k):

    e = y_i - (b0 + b1 * x_i)
    s2 = np.sum(w * e ** 2) / (k - 2)
    
    return s2

### Step 6: Calculate the variance of the expected value of the response variable given that $x$ = $x_0$ 

The calculation of the variance is implemented by using this formula $Var(y_0) = \sigma^2 \left( \frac{1}{{\sum_i w_i}} + \frac{(x_0 - \bar{x}_w)^2}{\sum_i (x_i - \bar{x}_w)^2} \right)$
, the weighted version of the one in the lecture notes. The unweighted version of this formula in the notes uses instead of $\frac{1}{{\sum_i w_i}}$ this $\frac{1}{k}$ or $\frac{1}{n}$. This change in the formula is because the $\sum_i w_i$ reflects the effective number of points contributing to the local fit after weighting them, since as we said points far from $x_0$ get small weights and contribute less, so using k instead would understate the true uncertainty/variance.

For the exercise we want to find the standard error of the expected value of each predicted value, so we should convert the variance to $se$ by square-rooting the variance value.

In [17]:
def se_expected_value(x_i, x0, w, sigma2):

    x_bar_w = np.sum(w * x_i) / np.sum(w) # explained on step 4
    Sxx_w = np.sum(w * (x_i - x_bar_w) ** 2)
    sum_w = np.sum(w)

    var = sigma2 * (1 / sum_w + (x0 - x_bar_w) ** 2 / Sxx_w)

    return np.sqrt(var) # return the standard error which is the square root of the variance

### Step 7: Combine all the previous steps to create a function for performing predictions with Local Regression using one predictor

This step is for making the workflow reproducible.

In [18]:
def local_regression_weighted(y, x, k, x0):
    x = np.asarray(x)
    y = np.asarray(y)
    x0 = np.asarray(x0)

    pred = np.zeros(len(x0))
    se = np.zeros(len(x0))

    for i, x0_i in enumerate(x0):
        idx, dists = find_k_nearest_points(x, x0_i, k)
        x_i = x[idx]
        y_i = y[idx]

        w = assign_weights(dists)

        b0, b1 = weighted_OLS(x_i, y_i, w)
        pred[i] = b0 + b1 * x0_i

        sigma2 = error(x_i, y_i, w, b0, b1, k)
        se[i] = se_expected_value(x_i, x0_i, w, sigma2)

    return pred, se

### Step 8: Use the *local_regression_weighted* function to make predictions for an area with 10, 18 and 25 % of families with income < $3000

In [20]:
x0_values = [10, 18, 25]
k = 15 

pred, se = local_regression_weighted(MORT_y, POOR_x, k, x0_values)

results = pd.DataFrame({
    "POOR (x0)": x0_values,
    "Predicted MORT (pred)": pred,
    "se": se
})

print(results)

   POOR (x0)  Predicted MORT (pred)         se
0         10             901.722592  17.655681
1         18             958.683221  18.795556
2         25            1009.472614  24.272510


The k picked is random and as mentioned in the exercise a better choice of k can derived from cross-validation where many k's are used in order to find the most suitable/best performing one for this data.

| POOR (x0) | Predicted MORT (pred) | se |
|----------:|----------------------:|---:|
| 10        | 901.722592            | 17.655681 |
| 18        | 958.683221            | 18.795556 |
| 25        | 1009.472614           | 24.272510 |

The results for the $x_0$ values asked in the exercise, with k = 15.